In [1]:
import pandas as pd
import os
import datetime

# --- Logging Functions ---
LOG_FILE_PATH = "data_preparation_log.txt" # Changed log file name

def get_timestamp():
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def _log_to_file_and_console(level, message):
    log_entry = f"[{get_timestamp()} {level}] {message}\n"
    try:
        with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
            f.write(log_entry)
    except Exception as e:
        print(f"Failed to write to log file: {e}")
    print(log_entry.strip(), flush=True)

def log_info(message): _log_to_file_and_console("INFO", message)
def log_warn(message): _log_to_file_and_console("WARN", message)
def log_error(message): _log_to_file_and_console("ERROR", message)

def main():
    # Clear log file at the start of a new run
    if os.path.exists(LOG_FILE_PATH):
        try:
            os.remove(LOG_FILE_PATH)
            print(f"[{get_timestamp()} INIT] Cleared old log file: {LOG_FILE_PATH}")
        except Exception as e_rm_log:
            print(f"[{get_timestamp()} INIT] WARN: Could not clear old log file {LOG_FILE_PATH}: {e_rm_log}")

    project_split_file = "project_split_overall_details_best_ep16_project_split_2.csv"
    mutation_data_file = "mutation_data_with_spat_8222_1569.csv"
    output_csv_file = "complete_merged_dataset_2.csv" # New output file name

    log_info("Script started: Data Merging Only.")
    try:
        df_split = pd.read_csv(project_split_file)
        log_info(f"Successfully loaded {project_split_file}. Shape: {df_split.shape}")
        df_mutation = pd.read_csv(mutation_data_file)
        log_info(f"Successfully loaded {mutation_data_file}. Shape: {df_mutation.shape}")
    except FileNotFoundError as e:
        log_error(f"Input CSV file not found: {e.filename}"); return
    except Exception as e:
        log_error(f"Error reading input CSV files: {e}"); return

    # --- Define Merge Keys and Columns to Add ---
    # Keys used for joining the two dataframes
    merge_keys = [
        "project_name", 
        "full_test_name", 
        "source_file", 
        "augmented_source_marker", 
        "is_mutation", 
        "mutation_iteration"
    ]

    # Columns to be added from df_mutation to df_split
    # This now includes "pr_link" and other previously used columns
    cols_to_add_from_mutation = [
        "repo_url", 
        "original_sha", 
        "module", 
        "flaky_code", 
        "fixed_code", 
        "generated_patch",
        "pr_link" # Added pr_link
    ]

    log_info(f"df_split columns before processing: {df_split.columns.tolist()}")
    log_info(f"df_mutation columns before processing: {df_mutation.columns.tolist()}")

    # --- Prepare DataFrames for Merge ---
    # Ensure merge key columns are of string type and consistent for robust merging
    for key in merge_keys:
        # Check if key exists before trying to process it
        if key in df_split.columns:
            # Handle cases where a column might be all NaN, then convert to string
            if df_split[key].isnull().all():
                df_split[key] = "N/A_EMPTY_COL_SPLIT" # Placeholder for entirely NaN columns
            else:
                df_split[key] = df_split[key].astype(str)
            df_split[key] = df_split[key].str.strip().str.lower()
            if key == "is_mutation":
                df_split[key] = df_split[key].str.upper() # 'true'/'false' to 'TRUE'/'FALSE'
        else:
            log_warn(f"Merge key '{key}' not found in {project_split_file}. This key will be ignored for merging if not in both.")

        if key in df_mutation.columns:
            if df_mutation[key].isnull().all():
                df_mutation[key] = "N/A_EMPTY_COL_MUTATION"
            else:
                df_mutation[key] = df_mutation[key].astype(str)
            df_mutation[key] = df_mutation[key].str.strip().str.lower()
            if key == "is_mutation":
                df_mutation[key] = df_mutation[key].str.upper()
        else:
            log_warn(f"Merge key '{key}' not found in {mutation_data_file}. This key will be ignored for merging if not in both.")

    # Determine actual common keys present in both DataFrames for the merge operation
    actual_on_keys = [k for k in merge_keys if k in df_split.columns and k in df_mutation.columns]
    
    if not actual_on_keys:
        log_error("No common merge keys found after processing. Cannot merge dataframes.")
        return
    log_info(f"Actual keys used for merging: {actual_on_keys}")
    if len(actual_on_keys) < len(merge_keys):
        log_warn(f"Not all specified merge_keys were found in both dataframes. Missing: {set(merge_keys) - set(actual_on_keys)}")


    # Select only necessary columns from df_mutation: the merge keys + columns to add
    # This avoids merging all columns from df_mutation if not needed.
    cols_to_select_from_mutation = list(set(actual_on_keys + [col for col in cols_to_add_from_mutation if col in df_mutation.columns]))
    
    missing_cols_in_mutation = set(cols_to_add_from_mutation) - set(df_mutation.columns)
    if missing_cols_in_mutation:
        log_warn(f"The following requested columns to add are MISSING from {mutation_data_file}: {missing_cols_in_mutation}")
    
    log_info(f"Columns selected from {mutation_data_file} for merge: {cols_to_select_from_mutation}")


    # --- Perform the Merge ---
    try:
        # Using a left merge to keep all rows from df_split and add matching data from df_mutation
        df_complete = pd.merge(df_split, df_mutation[cols_to_select_from_mutation], 
                               on=actual_on_keys, how="left", suffixes=('', '_mut'))
        
        log_info(f"Merged dataframe shape: {df_complete.shape}. Total rows: {len(df_complete)}")

        # Check for duplicate columns that might have resulted from the merge if suffixes weren't perfect
        # (though with suffixes=('', '_mut'), this should be handled if there were original overlaps beyond keys)
        # If a column exists in both and is NOT a merge key, df_mutation's version will be suffixed.
        # Example: if df_split has 'repo_url' and df_mutation also has 'repo_url' (and it's not a merge key)
        # then after merge, you might have 'repo_url' (from df_split) and 'repo_url_mut' (from df_mutation).
        # We've explicitly selected columns from df_mutation, so this shouldn't create unexpected duplicates
        # unless `cols_to_add_from_mutation` also existed in `df_split` (which is fine, left merge prioritizes left).
        
        # Verify if new columns were added
        added_cols_check = [col for col in cols_to_add_from_mutation if col in df_complete.columns and col not in df_split.columns]
        log_info(f"Successfully added columns from mutation data: {added_cols_check}")
        if "pr_link" in df_complete.columns:
            log_info(f"Column 'pr_link' is present in the merged DataFrame.")
            log_info(f"Sample of 'pr_link' (first 5 non-NaN): \n{df_complete['pr_link'].dropna().head()}")
        else:
            log_warn(f"Column 'pr_link' is MISSING from the final merged DataFrame.")


    except pd.errors.MergeError as e:
        log_error(f"Pandas MergeError: {e}")
        log_error("This often happens if the merge keys are not properly aligned or have problematic data types despite attempts to standardize.")
        log_error(f"df_split dtypes for keys: {df_split[actual_on_keys].dtypes}")
        log_error(f"df_mutation dtypes for keys: {df_mutation[actual_on_keys].dtypes}")
        return
    except KeyError as e:
        log_error(f"KeyError during merge or column selection: {e}. This means a specified column name was not found.")
        return
    except Exception as e:
        log_error(f"An unexpected error occurred during merge: {e}")
        return

    if df_complete.empty:
        log_error("Merged dataframe is empty. Please check merge keys and input data.")
        # Log sample data from merge key columns to help diagnose
        for key in actual_on_keys:
            log_info(f"Sample of '{key}' in df_split (first 5): \n{df_split[key].head().to_string()}")
            log_info(f"Sample of '{key}' in df_mutation (first 5): \n{df_mutation[key].head().to_string()}")
        return
        
    # --- Save the Complete Dataset ---
    try:
        df_complete.to_csv(output_csv_file, index=False)
        log_info(f"Successfully saved the complete merged dataset to: {output_csv_file}")
    except Exception as e:
        log_error(f"Error saving the merged dataset: {e}")

    log_info("Data preparation and merging complete.")

if __name__ == "__main__":
    # Clear log file at the start of a new run
    if os.path.exists(LOG_FILE_PATH):
        try:
            os.remove(LOG_FILE_PATH)
            # Use print here as logger might not be fully set up if main() itself has issues
            print(f"[{get_timestamp()} INIT] Cleared old log file: {LOG_FILE_PATH}")
        except Exception as e_rm_log:
            print(f"[{get_timestamp()} INIT] WARN: Could not clear old log file {LOG_FILE_PATH}: {e_rm_log}")
    main()

[2025-05-25 23:42:51 INIT] Cleared old log file: data_preparation_log.txt
[2025-05-25 23:42:51 INFO] Script started: Data Merging Only.
[2025-05-25 23:42:51 INFO] Successfully loaded project_split_overall_details_best_ep16_project_split_2.csv. Shape: (336, 13)
[2025-05-25 23:42:51 INFO] Successfully loaded mutation_data_with_spat_8222_1569.csv. Shape: (1544, 24)
[2025-05-25 23:42:51 INFO] df_split columns before processing: ['project_name', 'full_test_name', 'source_file', 'augmented_source_marker', 'isCorrect', 'incorrect_type', 'is_mutation', 'mutation_iteration', 'model_predicted_prob', 'model_predicted_label_thresh0.5', 'rerun_consistency', 'prediction_time_secs', 'rerun_time_consume']
[2025-05-25 23:42:51 INFO] df_mutation columns before processing: ['repo_url', 'repo_owner', 'project_name', 'original_sha', 'fixed_sha', 'module', 'full_test_name', 'pr_link', 'flaky_code', 'fixed_code', 'diff', 'generated_patch', 'isCorrect', 'incorrect_type', 'incorrect_reason', 'rerun_consistency